In [ ]:
import numpy as np
import pandas as pd
import os


## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']


## Select the test
test= 'PUB'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

if model=='HBV':
    dyn= 'parBETA_parBETAET_'

if model=='marrmot_PRMS':
    dyn= 'alpha_cgw_scx_tt_'

if model=='HBV_capillary':
    dyn= 'parBETA_parBETAET_parK0_parUZL_'

if test== 'PUB':
    n=11
if test== 'PUR':
    n=8

 

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den

    
locations= pd.read_csv("/home/s0shar47/scratch/salil/DPL/dPL_Hydro_SNTEMP/config/gauge_loc.csv")

metrics = {
    "GageID": [],
    "NSE-Flow (Single-Objective)": [],
    "NSE-Flow (Multi-Objective)": [],
    "NSE-Baseflow (Single-Objective)": [],
    "NSE-Baseflow (Multi-Objective)": [],
    "lat": [],
    "lon": []
}

for i in range(1, n):

    if test== 'PUB':
        folder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[1]}/{model}/{i}/'
        sfolder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[0]}/{model}/{i}/'
        gauge = np.load(f"/home/s0shar47/scratch/BASEFLOW_FOLDS/{i}/test/DAYMET_19800101_20141231_forcing_baseflow_mmday.npy")
    
 
      

    ob_flow = np.load(
        os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/00060_Mean.npy")
    )

    sim_flow_old = np.load(
        os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy"))

    sim_flow_new = np.load(
        os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy")
    )

    ob_baseflow = np.load(
        os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/baseflow.npy")
    )

    sim_gwflow_old = np.load(
        os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
    )

    sim_gwflow_new = np.load(
        os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
    )

    sim_flow_old = np.squeeze(sim_flow_old)
    sim_flow_new = np.squeeze(sim_flow_new)
    sim_gwflow_old = np.squeeze(sim_gwflow_old)
    sim_gwflow_new = np.squeeze(sim_gwflow_new)

    # --------------------------------------------------------
    # Loop over basins
    # --------------------------------------------------------
    for k in range(ob_flow.shape[1]):

        

        gid = int(gauge[k, 0, -2])

        matching_row = locations[locations['GAGE_ID'] == gid]
        lat = matching_row['LAT'].item()
        lon = matching_row['LONG'].item()
        # lat= locations.loc[f'{gid}',['LAT']]
        # lon= locations.loc[f'{gid}',['LONG']]

        metrics["GageID"].append(gid)
        metrics["lat"].append(lat)
        metrics["lon"].append(lon)

        ## Baseflow
        obs = ob_baseflow[:, k]
        old = sim_gwflow_old[:, k]
        new = sim_gwflow_new[:, k]

        # =====================================================
        # Overall metrics
        # =====================================================
        metrics["NSE-Baseflow (Single-Objective)"].append(calc_nse(obs, old))
        metrics["NSE-Baseflow (Multi-Objective)"].append(calc_nse(obs, new))

        ## Flow
        obs = ob_flow[:, k]
        old = sim_flow_old[:, k]
        new = sim_flow_new[:, k]

        # =====================================================
        # Overall metrics
        # =====================================================
        metrics["NSE-Flow (Single-Objective)"].append(calc_nse(obs, old))
        metrics["NSE-Flow (Multi-Objective)"].append(calc_nse(obs, new))
table = pd.DataFrame(metrics)
table

In [ ]:
import json
import numpy as np

# Load the NumPy array
bfi = np.load(
    "/home/s0shar47/scratch/Daymet_Folds/1/test/daymet_19800101_20141231_forcing_671.npy"
)

# Open and load the JSON metadata/names file
json_path = "/home/s0shar47/scratch/Daymet_Folds/1/test/daymet_19800101_20141231_forcing_671_name.json"
with open(json_path, "r") as f:
    feature_names = json.load(f)

# Inspect the loaded data
print(f"NumPy Array Shape: {bfi.shape}")
print(f"NumPy Array Data Type: {bfi.dtype}")
print(f"Loaded JSON Content: {feature_names}")

In [ ]:
###Plot1 Figure of CAMELS basins

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

plt.rcParams['font.family']= 'Nimbus Roman'

gdf = gpd.read_file("/scratch/data/cb_2018_us_nation_20m/cb_2018_us_nation_20m.shp")

# Make sure the layer is in geographic coordinates (WGS84)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# Define CONUS bounding box: minx, miny, maxx, maxy
conus_bbox = box(-125.0, 24.0, -66.5, 50.0)

# Clip the geodataframe to the bounding box
gdf_conus = gpd.clip(gdf, conus_bbox)

# Create the plot figure
fig, ax = plt.subplots(figsize=(14, 14))

# 1. Plot the map outline backdrop first
gdf_conus.boundary.plot(ax=ax, edgecolor="black", linewidth=2, zorder=1)

# # 2. Plot Single-Objective performance markers (Square shape)
# scatter1 = ax.scatter(
#     table['lon'], table['lat'], 
#     marker='s', s=80, 
#     c=table['NSE-Flow (Single-Objective)'], 
#     cmap='viridis', vmin=0, vmax=1,  # Set color limits for clear NSE comparison
#     label='Single-Objective', zorder=2
# )

# 3. Plot Multi-Objective performance markers (Circle shape)
scatter2 = ax.scatter(
    table['lon'], table['lat'], 
    marker='o', s=30, 
    color= 'red' 
    ,label='Basins'
)


# 5. Finalize legend and axis view configurations
# plt.legend(loc='upper right', fontsize=15)
ax.set_title("CAMELS Basins", fontsize=18, fontweight='bold', pad=15)
# ax.set_axis_off()  # Optional: hides degree grid lines for a cleaner look
plt.xlabel('Longitude(°E)', fontsize=15)
plt.ylabel('Latitude(°N)', fontsize=15)
plt.grid()
plt.show()


In [ ]:
## Plot2 Scatter Plot Showing advantages of Multi-objective
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']

method= 'LocalHYSEP'
methods= ['USGS', 'Furey', 'Eckhardt', 'LocalHYSEP']

## Select the test
test= 'PUR'
tests= ['PUR','PUB']

## Select the model
model= 'HBV'
models= ['HBV', 'marrmot_PRMS', 'HBV_capillary']

for test in tests:
    for method in methods:
        for model in models:

            if model=='HBV':
                dyn= 'parBETA_parBETAET_'

            if model=='marrmot_PRMS':
                dyn= 'alpha_cgw_scx_tt_'

            if model=='HBV_capillary':
                dyn= 'parBETA_parBETAET_parK0_parUZL_'

            if test== 'PUB':
                n=11
            if test== 'PUR':
                n=8


            df= pd.read_excel("/home/s0shar47/scratch/salil/GAGES-2/gagesII_mar1_2013_conterm.xlsx",sheet_name='Hydro')
            df= pd.DataFrame(df.set_index('STAID')['BFI_AVE'])
            df.reset_index(inplace=True)

            bfi_sim = []
            gauges = []

            bfi_sim_old= [] 

            for i in range(1,n):
                if test== 'PUB':
                            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/{test}/{method}_{objectives[1]}//{i}/"
                            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/{test}/OUTPUTS_PUB_{objectives[0]}/{i}/"
                            gauge = np.load(
                                f"/scratch/data/PUB_DATA/test/{i}/daymet_19800101_20141231_forcing_671_baseflow_mmday.npy")
                
                
                if test== 'PUR':
                            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/{method}_{objectives[1]}/Region_{i}/"
                
                            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/OUTPUTS_PUR_{objectives[0]}/Region_{i}/"

                            gauge = np.load(
                                        f"/scratch/data/PUR_DATA/test/Region_{i}/daymet_19800101_20141231_forcing_671_baseflow_mmday.npy")

                # Load gage ids
                gage_id = gauge[:, 0, -2].astype(int)  # Convert to integer right away

                # Load simulated BFI
                bfim = np.load(
                    os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999//BFI_sim.npy"
                            ))

                bfio= np.load(
                            os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999///BFI_sim.npy"
                            ))

                # Append the raw numpy arrays
                gauges.append(gage_id)
                bfi_sim.append(bfim)
                bfi_sim_old.append(bfio)

            # Concatenate all 10 folds into flat 1D numpy arrays
            flat_gauges = np.concatenate(gauges)
            flat_bfi = np.concatenate(bfi_sim)
            flat_bfio = np.concatenate(bfi_sim_old)

            # Create a clean, flat DataFrame
            dfd = pd.DataFrame({"STAID": flat_gauges, "Simulated BFI (Multi)": flat_bfi, "Simulated BFI (Single)": flat_bfio})

            dff=dfd.merge(df, on='STAID', how='inner')

            # Extract arrays
            x = dff['BFI_AVE'].values
            y1 = dff['Simulated BFI (Single)'].values
            y2 = dff['Simulated BFI (Multi)'].values

            corr1= dff['BFI_AVE'].corr(dff['Simulated BFI (Single)'])
            rmse1= np.sqrt(((dff['BFI_AVE']-dff['Simulated BFI (Single)'])**2).mean())
            corr2= dff['BFI_AVE'].corr(dff['Simulated BFI (Multi)'])
            rmse2= np.sqrt(((dff['BFI_AVE']-dff['Simulated BFI (Multi)'])**2).mean())

            plt.figure(figsize=(11,11))

            # Scatter plots
            plt.scatter(x, y1, marker='o', label= f'Simulated BFI (Single-Objective)\nR:{corr1:.2f}\nRMSE: {rmse1:.2f}', color='red')
            plt.scatter(x, y2, marker='s', label= f'Simulated BFI (Multi-Objective)\nR:{corr2:.2f}\nRMSE: {rmse2:.2f}', color='blue')

            # 1:1 line
            min_val = min(np.min(x), np.min(y1), np.min(y2))
            max_val = max(np.max(x), np.max(y1), np.max(y2))
            plt.plot([0, 100], [0, 100], 'k--', label='1:1 line', linewidth=2)

            # Labels and styling
            plt.xlim(0,100)
            plt.ylim(0,100)
            plt.xlabel('Observed BFI', fontsize=23)
            plt.xticks(fontsize=22)
            plt.ylabel('Simulated BFI', fontsize=23)
            plt.yticks(fontsize=22)
            # plt.title('BFI: Observed vs Simulated(Timeline 1980-2014)', fontsize=18, fontweight='bold')
            plt.title(f'BFI: Observed vs Simulated\n{method}, {test}, {model}', fontsize=18, fontweight='bold')
            plt.legend(
                fontsize=22,
                markerscale=2.5,   # Increase marker size in legend
                scatterpoints=1
            )
            plt.grid(True)

            plt.tight_layout()
            plt.savefig(f"/home/s0shar47/scratch/OUTPUTS_Paper1/Figures/BFI_SCATTER/{method}_{test}_{model}.jpg", dpi=300)
            plt.show()

In [2]:
## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']

method= 'Furey'
methods= ['USGS', 'Furey', 'Eckhardt', 'LocalHYSEP']
## Select the test
test= 'PUR'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

if model=='HBV':
    dyn= 'parBETA_parBETAET_'

if model=='marrmot_PRMS':
    dyn= 'alpha_cgw_scx_tt_'

if model=='HBV_capillary':
    dyn= 'parBETA_parBETAET_parK0_parUZL_'

if test== 'PUB':
    n=11
if test== 'PUR':
    n=8

    
import numpy as np
import os

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den


def calc_kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    std_obs = np.std(obs)
    std_sim = np.std(sim)

    if std_obs == 0:
        return np.nan

    alpha = std_sim / std_obs

    mean_obs = np.mean(obs)
    if mean_obs == 0:
        return np.nan

    beta = np.mean(sim) / mean_obs

    return 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )


def calc_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return (np.sum(sim) - den) / den


def calc_abs_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return np.abs(np.sum(sim) - den) / den


models= ['HBV', 'HBV_capillary', 'marrmot_PRMS']
# models= ['HBV', 'marrmot_PRMS']

for k in range(len(models)):
    model= models[k]

    if model=='HBV':
        dyn= 'parBETA_parBETAET_'

    if model=='marrmot_PRMS':
        dyn= 'alpha_cgw_scx_tt_'

    if model=='HBV_capillary':
        dyn= 'parBETA_parBETAET_parK0_parUZL_'

# ============================================================
# Metric functions
# ============================================================



# ============================================================
# Containers
# ============================================================

    metrics_old = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    metrics_new = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    # ============================================================
    # Loop over folds
    # ============================================================

    for i in range(1, n):


        if test== 'PUB':
            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/{method}_{objectives[1]}/{i}/"

            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/OUTPUTS_PUB_{objectives[0]}/{i}/"
            # gauge = np.load(
            #     f"/home/s0shar47/scratch/BASEFLOW_FOLDS/{i}/test/DAYMET_19800101_20141231_forcing_baseflow_mmday.npy")
 

        if test== 'PUR':
            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/{method}_{objectives[1]}/Region_{i}/"

            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/OUTPUTS_PUR_{objectives[0]}/Region_{i}/"


        obs_flow = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/00060_Mean.npy")
        )

        sim_old = np.load(
            os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy")
            # f"/home/s0shar47/scratch/OUTPUTS_PUR_Single/Region_{i}/LSTM_HBV_capillary_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_parK0_parUZL_/ts1989_1999/flow_sim.npy"

        )

        sim_new = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy")
            # f"/home/s0shar47/scratch/OUTPUTS_PUR_Multi/Region_{i}/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/flow_sim.npy"
            # f"/home/s0shar47/scratch/OUTPUTS_PUR_Multi/Region_{i}/LSTM_HBV_capillary_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_parK0_parUZL_/ts1989_1999/flow_sim.npy"
        )

        sim_old = np.squeeze(sim_old)
        sim_new = np.squeeze(sim_new)

        # --------------------------------------------------------
        # Loop over basins
        # --------------------------------------------------------
        for k in range(obs_flow.shape[1]):

            obs = obs_flow[:, k]
            old = sim_old[:, k]
            new = sim_new[:, k]

            # =====================================================
            # Overall metrics
            # =====================================================
            metrics_old["NSE"].append(calc_nse(obs, old))
            metrics_new["NSE"].append(calc_nse(obs, new))

            metrics_old["KGE"].append(calc_kge(obs, old))
            metrics_new["KGE"].append(calc_kge(obs, new))

            metrics_old["Bias"].append(calc_bias(obs, old))
            metrics_new["Bias"].append(calc_bias(obs, new))

            metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
            metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

            # =====================================================
            # Low / High flow
            # =====================================================
            threshold = np.median(obs)
            # print(threshold, np.mean(obs))

            low = obs <= threshold
            high = obs > threshold


            # print(f"Num_overall:{np.sum((obs - new) ** 2)}")
            # print(f"Den_overall:{np.sum((obs - np.mean(obs)) ** 2)}")
            # print(f"Overall Mean:{np.mean(obs)}")

            # print(f"Num_high:{np.sum((obs[high]- new[high])** 2)}")
            # print(f":Den_high:{np.sum((obs[high] - np.mean(obs[high])) ** 2)}")
            # print(f"High Mean:{np.mean(obs[high])}")


            # print(f"Num_low:{np.sum((obs[low] - new[low]) ** 2)}")
            # print(f"Den_low:{np.sum((obs[low] - np.mean(obs[low])) ** 2)}")
            # print(f"Low Mean:{np.mean(obs[low])}")


            # ---------------- Low flow ----------------
            if np.sum(low) > 2:

                metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
                metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))

                metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
                metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))

                metrics_old["AbsBias_low"].append(
                    calc_abs_bias(obs[low], old[low])
                )
                metrics_new["AbsBias_low"].append(
                    calc_abs_bias(obs[low], new[low])
                )

            # ---------------- High flow ----------------
            if np.sum(high) > 2:

                metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
                metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))

                metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
                metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))

                metrics_old["AbsBias_high"].append(
                    calc_abs_bias(obs[high], old[high])
                )
                metrics_new["AbsBias_high"].append(
                    calc_abs_bias(obs[high], new[high])
                )

    # ============================================================
    # Print comparison
    # ============================================================

    # print("\n" + "=" * 60)
    print(f"{test} Test TOTAL FLOW COMPARISON ({model} 1989-1999) using {method} for baseflow separation with median as threshold")
    # print("=" * 60)

    metric_order = [
        "NSE",
        "KGE",
        "Bias",
        "AbsBias",
        "NSE_low",
        "NSE_high",
        "Bias_low",
        "Bias_high",
        "AbsBias_low",
        "AbsBias_high",
    ]

    for metric in metric_order:
        old = np.nanmedian(metrics_old[metric])
        new = np.nanmedian(metrics_new[metric])
        print(f"{metric:<13}: {old:8.4f} → {new:8.4f}")

    print("=" * 60)
    

PUR Test TOTAL FLOW COMPARISON (HBV 1989-1999) using Furey for baseflow separation with median as threshold
NSE          :   0.6331 →   0.6345
KGE          :   0.6501 →   0.6574
Bias         :  -0.0340 →  -0.0538
AbsBias      :   0.1284 →   0.1231
NSE_low      :  -3.7524 →  -3.9197
NSE_high     :   0.5765 →   0.5652
Bias_low     :   0.3044 →   0.2529
Bias_high    :  -0.0698 →  -0.0770
AbsBias_low  :   0.4468 →   0.4502
AbsBias_high :   0.1431 →   0.1369
PUR Test TOTAL FLOW COMPARISON (HBV_capillary 1989-1999) using Furey for baseflow separation with median as threshold
NSE          :   0.6214 →   0.6358
KGE          :   0.6281 →   0.6553
Bias         :  -0.0372 →  -0.0575
AbsBias      :   0.1302 →   0.1310
NSE_low      :  -4.1002 →  -3.1722
NSE_high     :   0.5530 →   0.5733
Bias_low     :   0.3863 →   0.2533
Bias_high    :  -0.0687 →  -0.0888
AbsBias_low  :   0.4840 →   0.4376
AbsBias_high :   0.1408 →   0.1449
PUR Test TOTAL FLOW COMPARISON (marrmot_PRMS 1989-1999) using Furey for ba

In [2]:
obs_flow.shape

(3286, 56)

In [ ]:
import numpy as np
import os

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den

# def calc_nse(targetflat, predflat):
#     mask = np.isfinite(targetflat) & np.isfinite(predflat)

    # if np.sum(mask) < 2:
    #     return np.nan

    # target = targetflat[mask]
    # pred = predflat[mask]

    # den = np.sum((target - np.mean(target)) ** 2)
    # if den == 0:
    #     return np.nan

    # return 1 - (
    #     np.sum((pred - target) ** 2)
    #     / den
    # )

obs= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/USGS_Multi/Region_1/LSTM_marrmot_PRMS_E50_R365_B100_H256_tr1989_1999_n16_111111/alpha_cgw_scx_tt_/ts1989_1999/00060_Mean.npy")
sim= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/USGS_Multi/Region_1/LSTM_marrmot_PRMS_E50_R365_B100_H256_tr1989_1999_n16_111111/alpha_cgw_scx_tt_/ts1989_1999/flow_sim.npy")
sim = sim.squeeze()

calc_nse(obs,sim)

In [ ]:
threshold = np.median(obs)

low = obs <= threshold
high = obs > threshold
low.shape
obs[low]

In [ ]:
epsilon = 1e-6 
log_obs = np.log(obs[:, i] + epsilon)
log_sim = np.log(sim[:, i] + epsilon)

nse_low = calc_nse(log_obs, log_sim)
nse_low

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for i in range(4):
    threshold = np.median(obs[:,i])
    plt.figure(figsize=(14,5))
    low_sim= (sim[:,i][obs[:,i] <= threshold])
    low_obs= (obs[:,i][obs[:,i] <= threshold])
    # plt.plot(abs(sim[:,i]-obs[:,i]),color='red', label='absolute_error')
    plt.plot(obs[:,i],label= 'Observation', color='blue')
    plt.plot(sim[:,i],label= 'Simulation', color= 'red')
    plt.axhline(
        y=np.median(obs[:, i]), 
        color='green', 
        linestyle='--', 
        label=f'Obs Median ({np.median(obs[:, i]):.2f})'
    )
    plt.axhline(
            y=np.mean(obs[:, i]), 
            color='black', 
            linestyle='-', 
            label=f'Obs Mean ({np.median(obs[:, i]):.2f})'
        )
    
    plt.title(f'NSE:{calc_nse(obs[:,i],sim[:,i])} \nDen:{np.sum(((obs[:,i]) - np.mean(obs[:,i])) ** 2)} \nNum:{np.sum(((obs[:,i] - sim[:,i])) ** 2)}\n NSE_low:{calc_nse(low_obs,low_sim)}\nDen_low:{np.sum(((low_obs) - np.mean(low_obs)) ** 2)} \nNum_low:{np.sum(((low_obs - low_sim)) ** 2)}')
    # plt.ylim(-1, np.median(obs[:,i]+1))
    plt.xlim(0,730)
    plt.legend()
    plt.show()

In [ ]:
## Plot3 Box Plots (Total Flow) for every metric
import matplotlib.pyplot as plt
import pandas as pd
## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']


## Select the test
test= 'PUR'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

if model=='HBV':
    dyn= 'parBETA_parBETAET_'

if model=='marrmot_PRMS':
    dyn= 'alpha_cgw_scx_tt_'

if model=='HBV_capillary':
    dyn= 'parBETA_parBETAET_parK0_parUZL_'

if test== 'PUB':
    n=11
if test== 'PUR':
    n=8

    
import numpy as np
import os

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den


def calc_kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    std_obs = np.std(obs)
    std_sim = np.std(sim)

    if std_obs == 0:
        return np.nan

    alpha = std_sim / std_obs

    mean_obs = np.mean(obs)
    if mean_obs == 0:
        return np.nan

    beta = np.mean(sim) / mean_obs

    return 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )


def calc_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return (np.sum(sim) - den) / den


def calc_abs_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return np.abs(np.sum(sim) - den) / den


# models= ['HBV', 'HBV_capillary', 'marrmot_PRMS']
models= ['marrmot_PRMS']
for k in range(len(models)):
    model= models[k]

    if model=='HBV':
        dyn= 'parBETA_parBETAET_'

    if model=='marrmot_PRMS':
        dyn= 'alpha_cgw_scx_tt_'

    if model=='HBV_capillary':
        dyn= 'parBETA_parBETAET_parK0_parUZL_'

# ============================================================
# Metric functions
# ============================================================



# ============================================================
# Containers
# ============================================================

    metrics_old = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    metrics_new = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    # ============================================================
    # Loop over folds
    # ============================================================

    for i in range(1, n):


        if test== 'PUB':
            folder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[1]}/{model}/{i}/'
            sfolder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[0]}/{model}/{i}/'
            gauge = np.load(
                f"/home/s0shar47/scratch/BASEFLOW_FOLDS/{i}/test/DAYMET_19800101_20141231_forcing_baseflow_mmday.npy")
 

        if test== 'PUR':
            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/USGS_{objectives[1]}/Region_{i}/"
            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/OUTPUTS_PUR_{objectives[0]}/Region_{i}/"


        obs_flow = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/00060_Mean.npy")
        )

        sim_old = np.load(
            os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy")
        )

        sim_new = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy")
        )

        sim_old = np.squeeze(sim_old)
        sim_new = np.squeeze(sim_new)

        # --------------------------------------------------------
        # Loop over basins
        # --------------------------------------------------------
        for k in range(obs_flow.shape[1]):

            obs = obs_flow[:, k]
            old = sim_old[:, k]
            new = sim_new[:, k]

            # =====================================================
            # Overall metrics
            # =====================================================
            metrics_old["NSE"].append(calc_nse(obs, old))
            metrics_new["NSE"].append(calc_nse(obs, new))

            metrics_old["KGE"].append(calc_kge(obs, old))
            metrics_new["KGE"].append(calc_kge(obs, new))

            metrics_old["Bias"].append(calc_bias(obs, old))
            metrics_new["Bias"].append(calc_bias(obs, new))

            metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
            metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

            # =====================================================
            # Low / High flow
            # =====================================================
            threshold = np.median(obs)

            low = obs <= threshold
            high = obs > threshold

            # ---------------- Low flow ----------------
            if np.sum(low) > 2:

                metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
                metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))

                metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
                metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))

                metrics_old["AbsBias_low"].append(
                    calc_abs_bias(obs[low], old[low])
                )
                metrics_new["AbsBias_low"].append(
                    calc_abs_bias(obs[low], new[low])
                )

            # ---------------- High flow ----------------
            if np.sum(high) > 2:

                metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
                metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))

                metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
                metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))

                metrics_old["AbsBias_high"].append(
                    calc_abs_bias(obs[high], old[high])
                )
                metrics_new["AbsBias_high"].append(
                    calc_abs_bias(obs[high], new[high])
                )

    if model== 'HBV':
        metrics_new_hbv= metrics_new.copy()
        metrics_old_hbv= metrics_old.copy()

    if model=='marrmot_PRMS':
        metrics_new_prms= metrics_new.copy()
        metrics_old_prms= metrics_old.copy()

    if model=='HBV_capillary':
        metrics_new_hbvcap= metrics_new.copy()
        metrics_old_hbvcap= metrics_old.copy()

    metric_order = [
        "NSE",
        "KGE",
        "Bias",
        "AbsBias",
        "NSE_low",
        "NSE_high",
        "Bias_low",
        "Bias_high",
        "AbsBias_low",
        "AbsBias_high",
    ]



for metric in metric_order:

    plt.figure(figsize=(14,6))

    bp=plt.boxplot(
        [
            pd.DataFrame(metrics_old_hbv)[metric].dropna(), pd.DataFrame(metrics_new_hbv)["NSE"].dropna()

         ,pd.DataFrame(metrics_old_prms)[metric].dropna(), pd.DataFrame(metrics_new_prms)["NSE"].dropna(),

         pd.DataFrame(metrics_old_hbvcap)[metric].dropna(), pd.DataFrame(metrics_new_hbvcap)["NSE"].dropna()
         ],
        tick_labels=["Single-HBV", "Multi-HBV","Single-PRMS", "Multi-PRMS","Single-HBVcap", "Multi-HBVcap"],
        showfliers=False,
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2)
    )

    plt.ylabel(metric,fontsize=15)
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)
    plt.title(f"Comparison of {metric} for Total Flow")

    colors = [
        "#1f77b4",  # HBV Single
        "#ff7f0e",  # HBV Multi
        "#2ca02c",  # PRMS Single
        "#d62728",  # PRMS Multi
        "#9467bd",  # HBV_capillary Single
        "#8c564b",  # HBV_capillary Multi
    ]

    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)

    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
## Plot4 Box Plots (Baseflow) for every metric
import matplotlib.pyplot as plt
## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']


## Select the test
test= 'PUB'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

if model=='HBV':
    dyn= 'parBETA_parBETAET_'

if model=='marrmot_PRMS':
    dyn= 'alpha_cgw_scx_tt_'

if model=='HBV_capillary':
    dyn= 'parBETA_parBETAET_parK0_parUZL_'

if test== 'PUB':
    n=11
if test== 'PUR':
    n=8

    
import numpy as np
import os
import pandas as pd

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den


def calc_kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    std_obs = np.std(obs)
    std_sim = np.std(sim)

    if std_obs == 0:
        return np.nan

    alpha = std_sim / std_obs

    mean_obs = np.mean(obs)
    if mean_obs == 0:
        return np.nan

    beta = np.mean(sim) / mean_obs

    return 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )


def calc_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return (np.sum(sim) - den) / den


def calc_abs_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return np.abs(np.sum(sim) - den) / den


models= ['HBV', 'HBV_capillary', 'marrmot_PRMS']
# models= ['HBV']
for k in range(len(models)):
    model= models[k]

    if model=='HBV':
        dyn= 'parBETA_parBETAET_'

    if model=='marrmot_PRMS':
        dyn= 'alpha_cgw_scx_tt_'

    if model=='HBV_capillary':
        dyn= 'parBETA_parBETAET_parK0_parUZL_'

# ============================================================
# Metric functions
# ============================================================



# ============================================================
# Containers
# ============================================================

    metrics_old = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    metrics_new = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    # ============================================================
    # Loop over folds
    # ============================================================

    for i in range(1, n):


        if test== 'PUB':
                    folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/{method}_{objectives[1]}/{i}/"
        
                    sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/OUTPUTS_PUB_{objectives[0]}/{i}/"


        if test== 'PUR':
            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/{method}_{objectives[1]}/Region_{i}/"
            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/OUTPUTS_PUR_{objectives[0]}/Region_{i}/"


        obs_flow = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/baseflow.npy")
        )

        sim_old = np.load(
            os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
        )

        sim_new = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
        )

        sim_old = np.squeeze(sim_old)
        sim_new = np.squeeze(sim_new)

        # --------------------------------------------------------
        # Loop over basins
        # --------------------------------------------------------
        for k in range(obs_flow.shape[1]):

            obs = obs_flow[:, k]
            old = sim_old[:, k]
            new = sim_new[:, k]

            # =====================================================
            # Overall metrics
            # =====================================================
            metrics_old["NSE"].append(calc_nse(obs, old))
            metrics_new["NSE"].append(calc_nse(obs, new))

            metrics_old["KGE"].append(calc_kge(obs, old))
            metrics_new["KGE"].append(calc_kge(obs, new))

            metrics_old["Bias"].append(calc_bias(obs, old))
            metrics_new["Bias"].append(calc_bias(obs, new))

            metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
            metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

            # =====================================================
            # Low / High flow
            # =====================================================
            threshold = np.median(obs)

            low = obs <= threshold
            high = obs > threshold

            # ---------------- Low flow ----------------
            if np.sum(low) > 2:

                metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
                metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))

                metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
                metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))

                metrics_old["AbsBias_low"].append(
                    calc_abs_bias(obs[low], old[low])
                )
                metrics_new["AbsBias_low"].append(
                    calc_abs_bias(obs[low], new[low])
                )

            # ---------------- High flow ----------------
            if np.sum(high) > 2:

                metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
                metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))

                metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
                metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))

                metrics_old["AbsBias_high"].append(
                    calc_abs_bias(obs[high], old[high])
                )
                metrics_new["AbsBias_high"].append(
                    calc_abs_bias(obs[high], new[high])
                )

    if model== 'HBV':
        metrics_new_hbv= metrics_new.copy()
        metrics_old_hbv= metrics_old.copy()

    if model=='marrmot_PRMS':
        metrics_new_prms= metrics_new.copy()
        metrics_old_prms= metrics_old.copy()

    if model=='HBV_capillary':
        metrics_new_hbvcap= metrics_new.copy()
        metrics_old_hbvcap= metrics_old.copy()

    metric_order = [
        "NSE",
        "KGE",
        "Bias",
        "AbsBias",
        "NSE_low",
        "NSE_high",
        "Bias_low",
        "Bias_high",
        "AbsBias_low",
        "AbsBias_high",
    ]



for metric in metric_order:

    plt.figure(figsize=(14,6))

    bp=plt.boxplot(
        [pd.DataFrame(metrics_old_hbv)[metric].dropna(), pd.DataFrame(metrics_new_hbv)[metric].dropna()
         ,pd.DataFrame(metrics_old_prms)[metric].dropna(), pd.DataFrame(metrics_new_prms)[metric].dropna(),
         pd.DataFrame(metrics_old_hbvcap)[metric].dropna(), pd.DataFrame(metrics_new_hbvcap)[metric].dropna()],
        tick_labels=["Single-HBV", "Multi-HBV","Single-PRMS", "Multi-PRMS","Single-HBVcap", "Multi-HBVcap"],
        showfliers=False,
        patch_artist=True,
        medianprops=dict(color="black", linewidth=2)
    )

    plt.ylabel(metric,fontsize=15)
    plt.xticks(fontsize=13)
    plt.yticks(fontsize=13)

    plt.title(f"Comparison of {metric} for baseflow")

    colors = [
        "#1f77b4",  # HBV Single
        "#ff7f0e",  # HBV Multi
        "#2ca02c",  # PRMS Single
        "#d62728",  # PRMS Multi
        "#9467bd",  # HBV_capillary Single
        "#8c564b",  # HBV_capillary Multi
    ]

    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)

    plt.grid(axis="y", alpha=0.5)
    plt.grid(axis='x')
    plt.tight_layout()
    plt.show()

In [3]:
## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']

method= 'Furey'
methods= ['USGS', 'Furey', 'Eckhardt', 'LocalHYSEP']
## Select the test
test= 'PUR'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

if model=='HBV':
    dyn= 'parBETA_parBETAET_'

if model=='marrmot_PRMS':
    dyn= 'alpha_cgw_scx_tt_'

if model=='HBV_capillary':
    dyn= 'parBETA_parBETAET_parK0_parUZL_'

if test== 'PUB':
    n=11
if test== 'PUR':
    n=8

    
import numpy as np
import os

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den


def calc_kge(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    r = np.corrcoef(obs, sim)[0, 1]

    std_obs = np.std(obs)
    std_sim = np.std(sim)

    if std_obs == 0:
        return np.nan

    alpha = std_sim / std_obs

    mean_obs = np.mean(obs)
    if mean_obs == 0:
        return np.nan

    beta = np.mean(sim) / mean_obs

    return 1 - np.sqrt(
        (r - 1) ** 2 +
        (alpha - 1) ** 2 +
        (beta - 1) ** 2
    )


def calc_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return (np.sum(sim) - den) / den


def calc_abs_bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    den = np.sum(obs)
    if den == 0:
        return np.nan

    return np.abs(np.sum(sim) - den) / den


models= ['HBV', 'HBV_capillary', 'marrmot_PRMS']
for k in range(len(models)):
    model= models[k]

    if model=='HBV':
        dyn= 'parBETA_parBETAET_'

    if model=='marrmot_PRMS':
        dyn= 'alpha_cgw_scx_tt_'

    if model=='HBV_capillary':
        dyn= 'parBETA_parBETAET_parK0_parUZL_'

# ============================================================
# Metric functions
# ============================================================



# ============================================================
# Containers
# ============================================================

    metrics_old = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    metrics_new = {
        "NSE": [],
        "KGE": [],
        "Bias": [],
        "AbsBias": [],
        "NSE_low": [],
        "NSE_high": [],
        "Bias_low": [],
        "Bias_high": [],
        "AbsBias_low": [],
        "AbsBias_high": [],
    }

    # ============================================================
    # Loop over folds
    # ============================================================

    for i in range(1, n):


        if test== 'PUB':
            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/{method}_{objectives[1]}/{i}/"
                
            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/OUTPUTS_PUB_{objectives[0]}/{i}/"
 

        if test== 'PUR':
            folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/{method}_{objectives[1]}/Region_{i}/"
            sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/OUTPUTS_PUR_{objectives[0]}/Region_{i}/"


        obs_flow = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/baseflow.npy")
        )

        sim_old = np.load(
            os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
            # f"/home/s0shar47/scratch/OUTPUTS_PUR_Single/Region_{i}/LSTM_HBV_capillary_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_parK0_parUZL_/ts1989_1999/flow_sim.npy"

        )

        sim_new = np.load(
            os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
            # f"/home/s0shar47/scratch/OUTPUTS_PUR_Multi/Region_{i}/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/flow_sim.npy"
            # f"/home/s0shar47/scratch/OUTPUTS_PUR_Multi/Region_{i}/LSTM_HBV_capillary_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_parK0_parUZL_/ts1989_1999/flow_sim.npy"
        )

        sim_old = np.squeeze(sim_old)
        sim_new = np.squeeze(sim_new)

        # --------------------------------------------------------
        # Loop over basins
        # --------------------------------------------------------
        for k in range(obs_flow.shape[1]):

            obs = obs_flow[:, k]
            old = sim_old[:, k]
            new = sim_new[:, k]

            # =====================================================
            # Overall metrics
            # =====================================================
            metrics_old["NSE"].append(calc_nse(obs, old))
            metrics_new["NSE"].append(calc_nse(obs, new))

            metrics_old["KGE"].append(calc_kge(obs, old))
            metrics_new["KGE"].append(calc_kge(obs, new))

            metrics_old["Bias"].append(calc_bias(obs, old))
            metrics_new["Bias"].append(calc_bias(obs, new))

            metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
            metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

            # =====================================================
            # Low / High flow
            # =====================================================
            threshold = np.median(obs)

            low = obs <= threshold
            high = obs > threshold

            # ---------------- Low flow ----------------
            if np.sum(low) > 2:

                metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
                metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))

                metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
                metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))

                metrics_old["AbsBias_low"].append(
                    calc_abs_bias(obs[low], old[low])
                )
                metrics_new["AbsBias_low"].append(
                    calc_abs_bias(obs[low], new[low])
                )

            # ---------------- High flow ----------------
            if np.sum(high) > 2:

                metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
                metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))

                metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
                metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))

                metrics_old["AbsBias_high"].append(
                    calc_abs_bias(obs[high], old[high])
                )
                metrics_new["AbsBias_high"].append(
                    calc_abs_bias(obs[high], new[high])
                )

    # ============================================================
    # Print comparison
    # ============================================================

    # print("\n" + "=" * 60)
    print(f"{test} Test BASEFLOW COMPARISON ({model} 1989-1999) using {method} Baseflow Separation")
    # print("=" * 60)

    metric_order = [
        "NSE",
        "KGE",
        "Bias",
        "AbsBias",
        "NSE_low",
        "NSE_high",
        "Bias_low",
        "Bias_high",
        "AbsBias_low",
        "AbsBias_high",
    ]

    for metric in metric_order:
        old = np.nanmedian(metrics_old[metric])
        new = np.nanmedian(metrics_new[metric])
        print(f"{metric:<13}: {old:8.4f} → {new:8.4f}")

    print("=" * 60)


PUR Test BASEFLOW COMPARISON (HBV 1989-1999) using Furey Baseflow Separation
NSE          :   0.3997 →   0.6092
KGE          :   0.4391 →   0.5857
Bias         :   0.1347 →  -0.0219
AbsBias      :   0.3509 →   0.2551
NSE_low      :  -5.7408 →  -2.9399
NSE_high     :   0.0504 →   0.3726
Bias_low     :   0.6978 →   0.2630
Bias_high    :   0.0708 →  -0.0776
AbsBias_low  :   0.7427 →   0.5146
AbsBias_high :   0.3028 →   0.2347
PUR Test BASEFLOW COMPARISON (HBV_capillary 1989-1999) using Furey Baseflow Separation
NSE          :   0.3803 →   0.6148
KGE          :   0.3489 →   0.5810
Bias         :  -0.1522 →  -0.0308
AbsBias      :   0.3785 →   0.2780
NSE_low      :  -3.8482 →  -2.7373
NSE_high     :   0.0357 →   0.3840
Bias_low     :   0.4749 →   0.2379
Bias_high    :  -0.2188 →  -0.0767
AbsBias_low  :   0.6368 →   0.5144
AbsBias_high :   0.3666 →   0.2417
PUR Test BASEFLOW COMPARISON (marrmot_PRMS 1989-1999) using Furey Baseflow Separation
NSE          :   0.2435 →   0.5879
KGE          : 

In [ ]:
obs_flow.shape

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
gws= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/Furey_Multi/Region_2/LSTM_marrmot_PRMS_E50_R365_B100_H256_tr1989_1999_n16_111111/alpha_cgw_scx_tt_/ts1989_1999/baseflow.npy")
gws= gws.squeeze()
gwm= np.load("/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/USGS_Multi/Region_2/LSTM_marrmot_PRMS_E50_R365_B100_H256_tr1989_1999_n16_111111/alpha_cgw_scx_tt_/ts1989_1999/baseflow.npy")

gwm= gwm.squeeze()
for i in range(5):
    plt.figure(figsize=(14,5))
    plt.plot(gws[730:1095,i], label="Furey")
    plt.plot(gwm[730:1095,i], label="USGS")
    plt.legend()
    plt.show()


In [ ]:
method= 'Furey'

## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']

flood= np.load("/scratch/data/PUR_DATA/test/Region_1/CAMELS_return_period_LP3.npy")
## Select the test
test= 'PUR'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

if model=='HBV':
    dyn= 'parBETA_parBETAET_'

if model=='marrmot_PRMS':
    dyn= 'alpha_cgw_scx_tt_'

if model=='HBV_capillary':
    dyn= 'parBETA_parBETAET_parK0_parUZL_'

if test== 'PUB':
    folder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[1]}/{model}/1/'
    sfolder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[0]}/{model}/1/'
    gauge = np.load(
        "/home/s0shar47/scratch/BASEFLOW_FOLDS/1/test/DAYMET_19800101_20141231_forcing_baseflow_mmday.npy")
 

if test== 'PUR':
    folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/{method}_{objectives[1]}/Region_1/"
    sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/OUTPUTS_PUR_{objectives[0]}/Region_1/"
    gauge= np.load(f"/scratch/data/PUR_DATA/test/Region_1/daymet_19800101_20141231_forcing_671_baseflow_mmday.npy")


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates
from pandas import Timedelta
import os

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den



# ----------------------------------------
# Plot first 10 basins
# ----------------------------------------
for i in range(14,21):
        # ----------------------------------------
    
    ob_flow= np.load(os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/00060_Mean.npy"))

    sim_flow_old= np.load(os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy"))

    sim_flow_new= np.load(os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/flow_sim.npy"))
    # sim_flow_new= sim_flow_new.squeeze()


    fig, ax = plt.subplots(figsize=(14, 5))

    gid = int(gauge[i, 0, -2])

    obs = ob_flow[0:730, i].copy()
    sim_old = sim_flow_old[0:730, i].copy()
    sim_new = sim_flow_new[0:730, i].copy()

    # ----------------------------------------
    # Date axis
    # ----------------------------------------
    dates = pd.date_range(
    start="1990-10-01",
    periods=obs.shape[0],
    freq="D"
)


    # den= np.sum((obs - obs.mean()) ** 2)
    # num1= np.sum((obs - sim_old) ** 2)
    # num2= np.sum((obs - sim_new) ** 2)

    # nse1= 1- num1/den
    # nse2= 1-num2/den
    sim_old= sim_old.squeeze()
    sim_new= sim_new.squeeze()

    nse1= calc_nse(obs,sim_old)
    nse2= calc_nse(obs,sim_new)

    # Replace non-positive values with NaN for log scale
    obs[obs <= 0] = np.nan
    sim_old[sim_old <= 0] = np.nan
    sim_new[sim_new <= 0] = np.nan

    # Observations as scatter
    ax.scatter(
        dates,
        obs,
        color="black",
        s=8,
        alpha=0.8,
        label="Observed Flow",
        zorder=3,
    )

    # Simulations as lines
    ax.plot(
        dates,
        sim_old,
        color="red",
        linewidth=1.5,
        label= f"Simulated Flow (Single-Objective)\nNSE: {nse1:.3f}",
        zorder=2,
    )

    ax.plot(
        dates,
        sim_new,
        color="blue",
        linewidth=1.5,
        label= f"Simulated Flow (Multi-Objective)\nNSE: {nse2:.3f}",
        zorder=2,
    )

    ax.plot(flood[i,0:730],label= 'Return Period', color='black')

    # Logarithmic y-axis
    ax.set_yscale("log")

    # Add padding
    buffer = Timedelta(days=10)
    ax.set_xlim(dates[0] - buffer, dates[-1] + buffer)

    # Year ticks
    # ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%b"))


    plt.xticks(rotation=45, fontsize= 13)
    plt.yticks(fontsize=13)

    ax.set_xlabel("Date", fontsize=15)
    ax.set_ylabel("Flow (mm/day)", fontsize=15)
    # ax.set_title(f"Streamflow Prediction\nUSGS Gauge: {gid}", fontsize=15)

    ax.grid(True, which="major", alpha=0.5)
    ax.grid(True, which="minor", alpha=0.2)

    ax.legend(loc="upper left",fontsize=12)
    print(f'{model}_{test}')
    print(f"Streamflow Prediction\nUSGS Gauge: {gid}")
    # plt.tight_layout()
    plt.show()


In [ ]:
## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']


## Select the test
test= 'PUB'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

if model=='HBV':
    dyn= 'parBETA_parBETAET_'

if model=='marrmot_PRMS':
    dyn= 'alpha_cgw_scx_tt_'

if model=='HBV_capillary':
    dyn= 'parBETA_parBETAET_parK0_parUZL_'

if test== 'PUB':
    folder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[1]}/{model}/1/'
    sfolder= f'/home/s0shar47/scratch/OUTPUTS_{test}_{objectives[0]}/{model}/1/'
    gauge = np.load(
        "/home/s0shar47/scratch/BASEFLOW_FOLDS/1/test/DAYMET_19800101_20141231_forcing_baseflow_mmday.npy")
 

if test== 'PUR':
    folder= f"/home/s0shar47/scratch/OUTPUTS_PUR_{objectives[1]}/Region_1/"
    sfolder= f"/home/s0shar47/scratch/OUTPUTS_PUR_{objectives[0]}/Region_1/"
    gauge= np.load(
        f"/home/s0shar47/scratch/PUR_DATA/test/Region_1/daymet_19800101_20141231_forcing_671_baseflow_mmday.npy")


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates
from pandas import Timedelta
import os

def calc_nse(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs[mask]
    sim = sim[mask]

    if len(obs) < 2:
        return np.nan

    den = np.sum((obs - np.mean(obs)) ** 2)
    if den == 0:
        return np.nan

    return 1 - np.sum((obs - sim) ** 2) / den



# ----------------------------------------
# Plot first 10 basins
# ----------------------------------------
for i in range(14,21):
        # ----------------------------------------
    
    ob_flow= np.load(os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/baseflow.npy"))

    sim_flow_old= np.load(os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy"))

    sim_flow_new= np.load(os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy"))
    # sim_flow_new= sim_flow_new.squeeze()


    fig, ax = plt.subplots(figsize=(14, 5))

    gid = int(gauge[i, 0, -2])

    obs = ob_flow[0:730, i].copy()
    sim_old = sim_flow_old[0:730, i].copy()
    sim_new = sim_flow_new[0:730, i].copy()

    # ----------------------------------------
    # Date axis
    # ----------------------------------------
    dates = pd.date_range(
    start="1990-10-01",
    periods=obs.shape[0],
    freq="D"
)


    # den= np.sum((obs - obs.mean()) ** 2)
    # num1= np.sum((obs - sim_old) ** 2)
    # num2= np.sum((obs - sim_new) ** 2)

    # nse1= 1- num1/den
    # nse2= 1-num2/den
    sim_old= sim_old.squeeze()
    sim_new= sim_new.squeeze()

    nse1= calc_nse(obs,sim_old)
    nse2= calc_nse(obs,sim_new)

    # Replace non-positive values with NaN for log scale
    obs[obs <= 0] = np.nan
    sim_old[sim_old <= 0] = np.nan
    sim_new[sim_new <= 0] = np.nan

    # Observations as scatter
    ax.scatter(
        dates,
        obs,
        color="black",
        s=8,
        alpha=0.8,
        label="Observed Flow",
        zorder=3,
    )

    # Simulations as lines
    ax.plot(
        dates,
        sim_old,
        color="red",
        linewidth=1.5,
        label= f"Simulated Flow (Single-Objective)\nNSE: {nse1:.3f}",
        zorder=2,
    )

    ax.plot(
        dates,
        sim_new,
        color="blue",
        linewidth=1.5,
        label= f"Simulated Flow (Multi-Objective)\nNSE: {nse2:.3f}",
        zorder=2,
    )

    # Logarithmic y-axis
    ax.set_yscale("log")

    # Add padding
    buffer = Timedelta(days=10)
    ax.set_xlim(dates[0] - buffer, dates[-1] + buffer)

    # Year ticks
    # ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%b"))


    plt.xticks(rotation=45)

    ax.set_xlabel("Date", fontsize=14)
    ax.set_ylabel("Flow (mm/day)", fontsize=14)
    # ax.set_title(f"Baseflow Prediction\nUSGS Gauge: {gid}", fontsize=14)

    ax.grid(True, which="major", alpha=0.5)
    ax.grid(True, which="minor", alpha=0.2)

    ax.legend(loc="upper left")

    print(f"{model}_test")

    print(f"Baseflow Prediction\nUSGS Gauge: {gid}")

    # plt.tight_layout()
    plt.show()



In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

gdf = gpd.read_file("/scratch/data/cb_2018_us_nation_20m/cb_2018_us_nation_20m.shp")

# Make sure the layer is in geographic coordinates (WGS84)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# Define CONUS bounding box: minx, miny, maxx, maxy
conus_bbox = box(-125.0, 24.0, -66.5, 50.0)

# Clip the geodataframe to the bounding box
gdf_conus = gpd.clip(gdf, conus_bbox)

# Create the plot figure
fig, ax = plt.subplots(figsize=(14, 14))

# 1. Plot the map outline backdrop first
gdf_conus.boundary.plot(ax=ax, edgecolor="black", linewidth=1, zorder=1)

# # 2. Plot Single-Objective performance markers (Square shape)
# scatter1 = ax.scatter(
#     table['lon'], table['lat'], 
#     marker='s', s=80, 
#     c=table['NSE-Flow (Single-Objective)'], 
#     cmap='viridis', vmin=0, vmax=1,  # Set color limits for clear NSE comparison
#     label='Single-Objective', zorder=2
# )

# 3. Plot Multi-Objective performance markers (Circle shape)
scatter2 = ax.scatter(
    table['lon'], table['lat'], 
    marker='o', s=80, 
    c=table['NSE-Baseflow (Multi-Objective)'], 
    cmap='Blues', vmin=0.2, vmax=1,
    label='Multi-Objective', zorder=3
)

# 4. Generate discrete colorbars for both objectives
# cbar1 = fig.colorbar(scatter1, ax=ax, orientation='horizontal', pad=0.05, shrink=0.4)
# cbar1.set_label('NSE Flow (Single-Objective)')

cbar2 = fig.colorbar(scatter2, ax=ax, orientation='horizontal', pad=0.09, shrink=0.8)
cbar2.set_label('NSE Baseflow (Multi-Objective)')

# 5. Finalize legend and axis view configurations
plt.legend(loc='upper right', fontsize=15)
ax.set_title("CONUS Streamflow Model Evaluation: NSE Baseflow Metrics", fontsize=18, fontweight='bold', pad=15)
# ax.set_axis_off()  # Optional: hides degree grid lines for a cleaner look
plt.xlabel('Longitude(°E)', fontsize=15)
plt.ylabel('Latitude(°N)', fontsize=15)
plt.grid()
plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

gdf = gpd.read_file("/scratch/data/cb_2018_us_nation_20m/cb_2018_us_nation_20m.shp")

# Make sure the layer is in geographic coordinates (WGS84)
if gdf.crs != "EPSG:4326":
    gdf = gdf.to_crs("EPSG:4326")

# Define CONUS bounding box: minx, miny, maxx, maxy
conus_bbox = box(-125.0, 24.0, -66.5, 50.0)

# Clip the geodataframe to the bounding box
gdf_conus = gpd.clip(gdf, conus_bbox)

# Create the plot figure
fig, ax = plt.subplots(figsize=(14, 14))

# 1. Plot the map outline backdrop first
gdf_conus.boundary.plot(ax=ax, edgecolor="black", linewidth=1, zorder=1)

# 2. Plot performance markers
scatter1 = ax.scatter(
    table['lon'], table['lat'], 
    marker='o', s=80, 
    c=(table['NSE-Flow (Multi-Objective)'] - table['NSE-Flow (Single-Objective)']), 
    cmap='seismic_r', 
    vmin=-0.4, 
    vmax=0.4,  
    label=r'$\text{NSE}_{\mathit{multi}} - \text{NSE}_{\mathit{single}}$', 
    zorder=2
)

# 4. Generate discrete colorbar
cbar1 = fig.colorbar(scatter1, ax=ax, orientation='horizontal', pad=0.09, shrink=0.8)
cbar1.set_label(r'$\Delta\text{NSE}\ (\text{NSE}_{\mathit{multi}} - \text{NSE}_{\mathit{single}})$', fontsize=14)
cbar1.ax.tick_params(labelsize=12)

# 5. Finalize legend and axis view configurations
leg = plt.legend(loc='upper right', fontsize=14, framealpha=1.0, facecolor='white', edgecolor='gray')

# FORCE LEGEND ICON COLOR TO BE VISIBLE (Changes it from invisible white to solid dark blue/grey)
for handle in leg.legend_handles:
    handle.set_color('#1f77b4')  # Solid blue color for the legend reference circle
    handle.set_sizes([100])      # Makes the legend circle slightly larger for readability

ax.set_title("CONUS Streamflow Model Evaluation: Change in NSE Flow Metrics", fontsize=18, fontweight='bold', pad=15)
plt.xlabel('Longitude (°E)', fontsize=15)  # Corrected to °W for CONUS coordinates
plt.ylabel('Latitude (°N)', fontsize=15)
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)
plt.grid()
plt.show()

In [ ]:
import os


## Select the number of objectives
objective= 'Multi'
objectives= ['Single', 'Multi']


## Select the test
test= 'PUB'
tests= ['PUB','PUR']

## Select the model
model= 'HBV'
models= ['HBV', 'PRMS', 'HBV_capillary']

    

    

In [ ]:
ob= np.load("/home/s0shar47/scratch/Outputs Multi Objective/HBV/1/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/baseflow.npy")

flow= np.load("/home/s0shar47/scratch/Outputs Multi Objective/HBV/1/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/gwflow.npy")
flow= flow.squeeze()



In [ ]:
import pandas as pd

# Function to create datetime
def make_datetime(yearmonth, day, time):
    year = (yearmonth // 100).astype(int)
    month = (yearmonth % 100).astype(int)

    # Pad time to 4 digits (e.g., 700 -> 0700, 5 -> 0005)
    time = time.fillna(0).astype(int).astype(str).str.zfill(4)

    hour = time.str[:2].astype(int)
    minute = time.str[2:].astype(int)

    return pd.to_datetime(
        {
            "year": year,
            "month": month,
            "day": day.astype(int),
            "hour": hour,
            "minute": minute,
        },
        errors="coerce",
    )

dfs=[]

for i in range(1996,2001):

    details= f"C:/Users/Salil Sharma/Downloads/StormEvents_details-ftp_v1.0_d{i}_c20260323.csv.gz"
    loc= f"C:/Users/Salil Sharma/Downloads/StormEvents_locations-ftp_v1.0_d{i}_c20260707.csv.gz"

    dt= pd.read_csv(details, compression="gzip")
    # dt= dt[dt['EVENT_TYPE']=='Flash Flood']

    lc= pd.read_csv(loc, compression='gzip')
    lc= lc.drop(columns={'LOCATION_INDEX', 'RANGE',
       'AZIMUTH', 'LAT2', 'LON2', 'EPISODE_ID'})

    # print(i)
    # print(dt.EVENT_TYPE.value_counts())
    df= pd.merge(dt,lc,on='EVENT_ID', how='inner')

    # Create start and end datetime columns
    df["START_DATETIME"] = make_datetime(
        df["BEGIN_YEARMONTH"],
        df["BEGIN_DAY"],
        df["BEGIN_TIME"],
    )

    df["END_DATETIME"] = make_datetime(
        df["END_YEARMONTH"],
        df["END_DAY"],
        df["END_TIME"],
    )

    # If you want separate date and time columns
    df["Start_Date"] = df["START_DATETIME"].dt.date
    df["Start_Time"] = df["START_DATETIME"].dt.strftime("%H:%M")

    df["End_Date"] = df["END_DATETIME"].dt.date
    df["End_Time"] = df["END_DATETIME"].dt.strftime("%H:%M")

    # Display
    # print(df[["START_DATE", "START_TIME", "END_DATE", "END_TIME"]].head())
    df= df.drop(columns={'BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME', 'END_YEARMONTH',
        'END_DAY', 'END_TIME', 'EPISODE_ID', 'EVENT_ID','STATE_FIPS',
        'YEAR', 'MONTH_NAME', 'CZ_TYPE', 'CZ_FIPS',
        'WFO', 'BEGIN_DATE_TIME', 'CZ_TIMEZONE', 'END_DATE_TIME',
        'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT',
        'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS', 'SOURCE',
        'MAGNITUDE', 'MAGNITUDE_TYPE',  'CATEGORY', 'TOR_F_SCALE',
        'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE',
        'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME', 'BEGIN_RANGE',
        'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH',
        'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
            'EVENT_NARRATIVE', 'YEARMONTH', 'START_DATETIME', 'END_DATETIME',
        })

    dfs.append(df)

df= pd.concat(dfs, ignore_index='True')
df
import numpy as np
df = df[df['CZ_NAME'].notna()]
df['EVENT_TYPE'].value_counts()